# Supply Chain Logistics & Demand Analysis Recipe

This recipe combines 3 `algebrax` tools to evaluate supply chain inventory allocation:

1. **Multidimensional Tensor Contraction** (`algebrax.trie.AlgebraicTrie`):
   Aggregates sparse 3D demand tensors `(Warehouse, Region, Season)` and contracts subtree totals using `trie.contract(prefix)`.
2. **Bounded Lattice Operations** (`algebrax.lattice.join` & `algebrax.lattice.meet`):
   Extracts peak capacity (Lattice Join $\vee$) and baseline safety stock (Lattice Meet $\wedge$) across product categories.
3. **Relative Entropy Divergence** (`algebrax.probability.kl_divergence`):
   Audits inventory allocation mismatch using KL divergence $D_{KL}(\text{Demand} \parallel \text{Supply})$.

In [ ]:
from algebrax.lattice import join, meet
from algebrax.probability import kl_divergence
from algebrax.semiring import StandardSemiring
from algebrax.trie import AlgebraicTrie

## 1. Multidimensional Tensor Contraction via AlgebraicTrie

We store 3D paths `(Warehouse, Region, Season) -> Volume` and contract subtree totals.

In [ ]:
trie = AlgebraicTrie(StandardSemiring)

trie.add((0, 101, 'Summer'), 500.0)
trie.add((0, 101, 'Winter'), 300.0)
trie.add((0, 102, 'Summer'), 200.0)
trie.add((1, 101, 'Summer'), 400.0)
trie.add((1, 103, 'Winter'), 600.0)

wh0_total = trie.contract((0,))
wh1_total = trie.contract((1,))

print(f'Contracted Total Demand for Warehouse 0: {wh0_total:.1f} units')
print(f'Contracted Total Demand for Warehouse 1: {wh1_total:.1f} units')

## 2. Bounded Lattice Peak & Baseline Capacity Bounds

We compute upper bound (Join $\vee$) and lower bound (Meet $\wedge$) demand vectors.

In [ ]:
category_a = {'Region_North': 1200.0, 'Region_South': 800.0, 'Region_East': 1500.0}
category_b = {'Region_North': 950.0, 'Region_South': 1100.0, 'Region_East': 1300.0}

peak_join = join(category_a, category_b)
baseline_meet = meet(category_a, category_b)

print('Peak Capacity Requirements (Lattice Join):      ', peak_join)
print('Baseline Safety Stock Requirements (Lattice Meet):', baseline_meet)

## 3. Inventory Allocation Divergence Audit (KL Divergence)

We calculate $D_{KL}(\text{Demand} \parallel \text{Supply}) = \sum d_i \ln \frac{d_i}{s_i}$.

In [ ]:
actual_demand = {'Region_North': 0.40, 'Region_South': 0.25, 'Region_East': 0.35}
inventory_alloc = {'Region_North': 0.30, 'Region_South': 0.30, 'Region_East': 0.40}

kl_score = kl_divergence(actual_demand, inventory_alloc)
print(f'Inventory Allocation Divergence D_KL(Demand || Supply): {kl_score:.6f} nats')